In [ ]:
import pandas as pd
import re
from itertools import product

def parse_prerequisites(prereq_text):
    """
    Parsea el texto de prerrequisitos y devuelve una lista de opciones.
    Cada opción es una lista de materias que deben cumplirse (AND).
    """
    if pd.isna(prereq_text) or prereq_text.strip() == '':
        return []
    
    # Limpiar el texto
    text = str(prereq_text).strip()
    
    # Reemplazar O y A por símbolos más fáciles de procesar
    text = text.replace(' O ', ' | ')
    text = text.replace(' A ', ' & ')
    text = text.replace(' Y', '')
    text = text.replace(' ', '')


    def evaluate_expression(expr):
        """
        Evalúa recursivamente expresiones con paréntesis
        """
        expr = expr.strip()
        temp_storage = {}
        temp_counter = 0
        
        # Procesar paréntesis de adentro hacia afuera
        while '(' in expr:
            # Encontrar el paréntesis más interno
            start = -1
            for i, char in enumerate(expr):
                if char == '(':
                    start = i
                elif char == ')' and start != -1:
                    end = i
                    # Procesar el contenido entre paréntesis
                    inner_content = expr[start+1:end]
                    inner_options = process_or_expression(inner_content)
                    
                    # Crear un placeholder temporal
                    temp_id = f"TEMP_{temp_counter}"
                    temp_storage[temp_id] = inner_options
                    temp_counter += 1
                    
                    # Reemplazar el paréntesis con el placeholder
                    expr = expr[:start] + temp_id + expr[end+1:]
                    break
            else:
                # Si no se encuentra ')', asume el final hasta el final de la línea
                if start != -1:
                    inner_content = expr[start+1:]
                    inner_options = process_or_expression(inner_content)
                    
                    temp_id = f"TEMP_{temp_counter}"
                    temp_storage[temp_id] = inner_options
                    temp_counter += 1
                    
                    expr = expr[:start] + temp_id
                    break
                    # Procesar el contenido entre paréntesis
                    inner_content = expr[start+1:i]
                    inner_options = process_or_expression(inner_content)
                    
                    # Crear un placeholder temporal
                    temp_id = f"TEMP_{temp_counter}"
                    temp_storage[temp_id] = inner_options
                    temp_counter += 1
                    
                    # Reemplazar el paréntesis con el placeholder
                    expr = expr[:start] + temp_id + expr[i+1:]
                    break
        
        # Procesar la expresión final
        final_options = process_or_expression(expr)
        
        # Expandir todos los placeholders
        expanded_options = expand_all_temps(final_options, temp_storage)
        
        return expanded_options
    
    def process_or_expression(expr):
        """
        Procesa expresiones dividiendo por OR (|)
        """
        # Dividir por OR
        or_parts = [part.strip() for part in expr.split('|')]
        result = []
        
        for part in or_parts:
            # Procesar AND
            and_parts = [p.strip() for p in part.split('&') if p.strip()]
            if and_parts:
                result.append(and_parts)
        
        return result
    
    def expand_all_temps(options, temp_storage):
        """
        Expande todos los placeholders temporales en todas las opciones
        """
        expanded_options = []
        
        for option in options:
            # Verificar si esta opción contiene placeholders
            temp_indices = []
            regular_parts = []
            
            for i, part in enumerate(option):
                if part.startswith('TEMP_') and part in temp_storage:
                    temp_indices.append((i, part))
                else:
                    regular_parts.append((i, part))
            
            if not temp_indices:
                # No hay placeholders, agregar la opción tal como está
                expanded_options.append(option)
            else:
                # Hay placeholders, necesitamos expandir todas las combinaciones
                expanded_combinations = expand_temp_combinations(option, temp_indices, temp_storage)
                expanded_options.extend(expanded_combinations)
        
        return expanded_options
    
    def expand_temp_combinations(base_option, temp_indices, temp_storage):
        """
        Expande todas las combinaciones posibles de los placeholders temporales
        """
        if not temp_indices:
            return [base_option]
        
        # Obtener todas las opciones para cada placeholder
        temp_options_list = []
        for idx, temp_id in temp_indices:
            temp_options_list.append(temp_storage[temp_id])
        
        # Generar todas las combinaciones posibles
        all_combinations = list(product(*temp_options_list))
        
        results = []
        for combination in all_combinations:
            # Crear una nueva opción reemplazando los placeholders
            new_option = base_option.copy()
            
            for i, (temp_idx, temp_id) in enumerate(temp_indices):
                # Reemplazar el placeholder con la opción seleccionada de la combinación
                selected_temp_option = combination[i]
                # Reemplazar el elemento en temp_idx con todos los elementos de selected_temp_option
                new_option[temp_idx:temp_idx+1] = selected_temp_option
            
            results.append(new_option)
        
        return results
    
    try:
        options = evaluate_expression(text)
        return options
    except Exception as e:
        print(f"Error procesando '{text}': {e}")
        return []

def process_prerequisites_file(file_path, courses_file_path=None):
    """
    Procesa el archivo de prerrequisitos y genera las nuevas columnas.
    Si se proporciona courses_file_path, filtra las opciones para que solo incluyan asignaturas válidas.
    """
    # Leer el archivo principal
    try:
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            df = pd.read_excel(file_path)
    except Exception as e:
        raise ValueError(f"Error leyendo el archivo: {e}")
    
    df= df[(df["Smbarul_Key_Rule"]=="FIS1033")]
    
    # Verificar que las columnas necesarias existen
    if 'prereq completo' not in df.columns:
        available_cols = list(df.columns)
        raise ValueError(f"La columna 'prereq completo' no se encuentra en el archivo. Columnas disponibles: {available_cols}")
    
    # Leer el archivo de asignaturas válidas si se proporciona
    valid_courses = None
    if courses_file_path:
        try:
            if courses_file_path.endswith('.csv'):
                df_courses = pd.read_csv(courses_file_path)
            else:
                df_courses = pd.read_excel(courses_file_path)
            if 'MatCrso' not in df_courses.columns:
                raise ValueError(f"La columna 'MatCrso' no se encuentra en el archivo de asignaturas.")
            # Filtrar las asignaturas válidas por el mismo programa académico
            if 'Programa' not in df_courses.columns:
                raise ValueError("La columna 'Programa' no se encuentra en el archivo de asignaturas.")
            if 'Programa' not in df.columns:
                raise ValueError("La columna 'Programa' no se encuentra en el archivo principal.")
            # Obtener el conjunto de programas presentes en el archivo principal
            programas_principal = set(df['Programa'].dropna().unique())
            # Filtrar df_courses solo por los programas presentes en el archivo principal
            df_courses_filtrado = df_courses[df_courses['Programa'].isin(programas_principal)]
            valid_courses = set(str(x).strip() for x in df_courses_filtrado['MatCrso'] if pd.notna(x))
        except Exception as e:
            raise ValueError(f"Error leyendo el archivo de asignaturas válidas: {e}")
    
    print("Procesando prerrequisitos...")
    
    # Procesar cada fila
    all_options = []
    max_options = 0
    
    for idx, row in df.iterrows():
        prereq_text = row['prereq completo']
        options = parse_prerequisites(prereq_text)
        print(row['Smbarul_Key_Rule'])
        
        # Convertir las opciones a strings con &
        formatted_options = []
        for option in options:
            if isinstance(option, list) and len(option) > 0:
                clean_option = [item.strip() for item in option if isinstance(item, str) and item.strip()]
                # Filtrar por asignaturas válidas si corresponde
                if valid_courses is not None:
                    # Todas las asignaturas de la opción deben estar en la lista
                    if all(course in valid_courses for course in clean_option):
                        if len(clean_option) > 1:
                            formatted_options.append(' & '.join(clean_option))
                        elif len(clean_option) == 1:
                            formatted_options.append(clean_option[0])
                    # Si no cumple, no se agrega la opción
                else:
                    if len(clean_option) > 1:
                        formatted_options.append(' & '.join(clean_option))
                    elif len(clean_option) == 1:
                        formatted_options.append(clean_option[0])
        # Si no hay opciones válidas, dejar vacío
        all_options.append(formatted_options)
        max_options = max(max_options, len(formatted_options))
        
        # Mostrar progreso cada 25 filas
        if (idx + 1) % 25 == 0:
            print(f"Procesadas {idx + 1} filas...")
    
    print(f"Creando {max_options} columnas de opciones...")
    
    # Crear las nuevas columnas
    for i in range(max_options):
        col_name = f'Opcion_Prereq_{i+1}'
        df[col_name] = ''
        for idx, options in enumerate(all_options):
            if i < len(options):
                df.loc[idx, col_name] = options[i]
    
    return df



In [4]:
def test_complex_case():
    """
    Prueba específica para el caso problemático
    """
    test_case = "( IIN4315 | IIN4319 ) & IST7122 & ( IGL7080 | IGL4925 | FRA7021 | IGL8530 | ALE7091 ) | IGL8525"
    
    print("Prueba del caso complejo:")
    print("=" * 80)
    print(f"Input: {test_case}")
    
    options = parse_prerequisites(test_case)
    
    print(f"Número de opciones generadas: {len(options)}")
    
    for i, option in enumerate(options, 1):
        if isinstance(option, list) and len(option) > 0:
            clean_option = [item.strip() for item in option if isinstance(item, str) and item.strip()]
            if len(clean_option) > 1:
                formatted = ' & '.join(clean_option)
            elif len(clean_option) == 1:
                formatted = clean_option[0]
            else:
                formatted = "Opción vacía"
        else:
            formatted = "Opción inválida"
        
        print(f"Opción {i}: {formatted}")
    
    print("=" * 80)

def show_examples():
    """
    Muestra ejemplos de cómo funciona el parser
    """
    examples = [
        "MAT1101 O INTE 00",
        "MAT1111 A ( FIS1023 O CSV0040 ) O INTE 00",
        "FIS1001 A MAT1001",
        "( QUI1001 O BIO1001 ) A MAT1002",
        "( IIN4315 | IIN4319 ) & IST7122 & ( IGL7080 | IGL4925 ) | IGL8525"
    ]
    
    print("Ejemplos de funcionamiento:")
    print("=" * 60)
    
    for example in examples:
        print(f"Input: '{example}'")
        options = parse_prerequisites(example)
        
        formatted_options = []
        for option in options:
            if isinstance(option, list) and len(option) > 0:
                clean_option = [item.strip() for item in option if isinstance(item, str) and item.strip()]
                if len(clean_option) > 1:
                    formatted_options.append(' & '.join(clean_option))
                elif len(clean_option) == 1:
                    formatted_options.append(clean_option[0])
        
        print(f"Opciones generadas: {len(formatted_options)}")
        for i, opt in enumerate(formatted_options, 1):
            print(f"  Opción {i}: {opt}")
        print("-" * 60)


In [3]:
def main():
    """
    Función principal para ejecutar el procesamiento
    """
    print("Procesador de Prerrequisitos Universitarios")
    print("=" * 50)
    
    # Opción para mostrar ejemplos o prueba compleja
    choice = 3 #input("¿Qué deseas hacer?\n1. Ver ejemplos\n2. Probar caso complejo\n3. Procesar archivo\nElige (1/2/3): ").strip()
    
    if choice == '1':
        show_examples()
        print()
    elif choice == '2':
        test_complex_case()
        print()
        return
    
    # Solicitar la ruta del archivo principal
    file_path = 'para trabajar pre req progv2.xlsx'#input("Ingresa la ruta del archivo principal de prerrequisitos (CSV o Excel): ").strip()
    
    if not file_path:
        print("No se proporcionó una ruta de archivo.")
        return None
    
    # Solicitar el archivo de asignaturas válidas
    courses_file_path ='asignaturas en plan de estudio.xlsx' #input("\nIngresa la ruta del archivo 'asignaturas en plan de estudio.xlsx' (o presiona Enter para omitir filtrado): ").strip()
    
    if not courses_file_path:
        print("⚠️ No se proporcionó archivo de asignaturas. Se procesarán todos los prerrequisitos sin filtrar.")
        courses_file_path = None
    
    try:
        # Procesar el archivo
        print(f"\nProcesando archivo: {file_path}")
        if courses_file_path:
            print(f"Archivo de asignaturas válidas: {courses_file_path}")
        
        df_result = process_prerequisites_file(file_path, courses_file_path)
        
        # Mostrar información sobre el resultado
        print(f"\n✅ Procesamiento completado!")
        print(f"Total de filas: {len(df_result)}")
        
        # Contar cuántas columnas de opciones se crearon
        option_cols = [col for col in df_result.columns if col.startswith('Opcion_Prereq_')]
        print(f"Columnas de opciones creadas: {len(option_cols)}")
        
        # Contar opciones válidas totales
        valid_options_count = 0
        for col in option_cols:
            valid_options_count += sum(1 for val in df_result[col] if val and str(val).strip())
        print(f"Total de opciones de prerrequisitos válidas: {valid_options_count}")
        
        # Mostrar filas con prerrequisitos (no vacías)
        non_empty_prereqs = df_result[df_result['prereq completo'].notna() & (df_result['prereq completo'] != '')]
        if len(non_empty_prereqs) > 0:
            print(f"\nFilas con prerrequisitos: {len(non_empty_prereqs)}")
            print("\nPrimeras 3 filas con prerrequisitos:")
            cols_to_show = []
            if 'Smbarul_Key_Rule' in df_result.columns:
                cols_to_show.append('Smbarul_Key_Rule')
            cols_to_show.extend(['prereq completo'] + option_cols[:5])  # Mostrar solo las primeras 5 columnas de opciones
            
            available_cols = [col for col in cols_to_show if col in df_result.columns]
            print(non_empty_prereqs[available_cols].head(3).to_string())
        
        # Guardar el resultado
        timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
        file_parts = file_path.rsplit('.', 1)
        if len(file_parts) == 2:
            output_path = file_parts[0] +timestamp+ '_procesado.' + file_parts[1]
        else:
            output_path = file_path +timestamp+ '_procesado.csv'
        
        if file_path.endswith('.csv') or not file_path.endswith(('.xlsx', '.xls')):
            df_result.to_csv(output_path, index=False)
        else:
            df_result.to_excel(output_path, index=False)
        
        print(f"\n💾 Archivo guardado como: {output_path}")
        
        return df_result
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

if __name__ == "__main__":
    main()

Procesador de Prerrequisitos Universitarios

Procesando archivo: para trabajar pre req progv2.xlsx
Archivo de asignaturas válidas: asignaturas en plan de estudio.xlsx
Procesando prerrequisitos...
CAS3020
ALE1031
ALE1031
MUS4110
ALE1031
ALE1031
POR4050
POR4050
POR4050
FRA1010
FRA1010
FRA1010
ALE1031
ALE1031
ALE1031
POR4050
POR4050
POR4050
POR4050
POR4050
POR4050
POR4050
POR4050
ALE1031
ALE1031
Procesadas 25 filas...
ALE1031
ALE1031
ALE1031
POR4050
POR4050
ALE1031
FRA1010
ALE1031
ALE1031
POR4050
POR4050
POR4050
POR4050
MAT1031
IBA0022
CAS3020
MAT1101
MAT1031
IIN0010
IBA0022
CAS3020
IME0010
MAT1101
MAT1031
ICI0010
Procesadas 50 filas...
IBA0022
CAS3020
EDU1050
CAS3020
PSI4401
ELG1120
EDU4529
ALE1031
MAT1101
MAT1031
IST2088
IST0010
CAS3020
MAT1101
FRA1010
FRA1010
ALE1031
ALE1031
POR4050
POR4050
FRA1010
POR4050
CAS3020
MUS4060
FRA1010
Procesadas 75 filas...
FRA1010
FRA1010
ALE1031
ALE1031
ALE1031
FRA1010
FRA1010
POR4050
POR4050
POR4050
POR4050
POR4050
POR4050
POR4050
FRA1010
ALE1031
ALE1031